In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
import cv2

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.layers import Layer, Dense, Conv2D, MaxPooling2D, GlobalAveragePooling2D, MaxPool2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, Adagrad, AdamW, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.utils.class_weight import compute_class_weight

In [ ]:
def f1_score_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(tf.math.round(y_pred), tf.float32)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum(y_pred) - tp
    fn = tf.reduce_sum(y_true) - tp
    precision = tp / (tp + fp + tf.keras.backend.epsilon())
    recall = tp / (tp + fn + tf.keras.backend.epsilon())
    f1_score = 2 * precision * recall / (precision + recall + tf.keras.backend.epsilon())
    return f1_score
model = load_model('/content/drive/MyDrive/Kartik /Mackerel(STACKED_MODEL)/mackerel2.h5', custom_objects={"f1_score_metric":f1_score_metric})

In [ ]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 densenet121 (Functional)    (None, 7, 7, 1024)        7037504   
                                                                 
 global_average_pooling2d (  (None, 1024)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 256)               262400    
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 5

In [ ]:
target_size=(224,224)
shape = (*target_size,3)
print(type(shape))
batch_size = 128

<class 'tuple'>


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=(0.7,1.3),
    fill_mode='nearest',
    validation_split = 0.3
)



path = '/content/drive/MyDrive/Kartik /Cropping/Mackerel/box/'

train_generator = train_datagen.flow_from_directory(
    directory=path,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    subset = 'training'
)

val_generator = train_datagen.flow_from_directory(
    directory=path,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle = False,
    subset = 'validation'
)

class_weights = compute_class_weight(
    'balanced', classes=np.unique(train_generator.classes), y=train_generator.classes
)
class_weights_dict = dict(zip(np.unique(train_generator.classes), class_weights))

# Print the computed class weights
print("Class Weights:", class_weights_dict)

Found 4823 images belonging to 2 classes.
Found 2065 images belonging to 2 classes.
Class Weights: {0: 1.2075613420130196, 1: 0.8533262561924982}


In [ ]:
from tensorflow.keras.metrics import Metric
class CustomPrecision(Metric):
    def __init__(self, name='custom_precision', **kwargs):
        super(CustomPrecision, self).__init__(name=name, **kwargs)
        self.precision = self.add_weight(name='precision', initializer='zeros')
        self.true_positives = self.add_weight(name='true_positives', initializer='zeros')
        self.false_positives = self.add_weight(name='false_positives', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_class1 = tf.argmax(y_pred, axis=-1)
        true_positives = tf.reduce_sum(tf.cast(tf.logical_and(tf.equal(y_true[:, 1], 1), tf.equal(y_pred_class1, 1)), dtype=tf.float32))
        false_positives = tf.reduce_sum(tf.cast(tf.logical_and(tf.equal(y_true[:, 0], 1), tf.equal(y_pred_class1, 1)), dtype=tf.float32))

        self.true_positives.assign_add(true_positives)
        self.false_positives.assign_add(false_positives)
        self.precision.assign(self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon()))

    def result(self):
        return self.precision

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras import backend as K
import tensorflow as tf
custom_optimizer = Adam()


import tensorflow as tf
from tensorflow.keras import backend as K

def f1_metric(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))

    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())

    f1 = 2 * (precision * recall) / (precision + recall + K.epsilon())
    return f1


def f1_loss(y_true, y_pred):
    epsilon = 1e-7
    y_pred = K.clip(y_pred, epsilon, 1 - epsilon)
    ce_loss = categorical_crossentropy(y_true, y_pred)
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)), axis=0)
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)), axis=0)
    actual_positives = K.sum(K.round(K.clip(y_true, 0, 1)), axis=0)
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (actual_positives + K.epsilon())
    f1 = 2 * (precision * recall) / (precision + recall + K.epsilon())
    f1_loss = -K.mean(f1)
    alpha = 0.4
    combined_loss = alpha * ce_loss + (1 - alpha) * f1_loss

    return combined_loss

# Usage in model compilation
model.compile(optimizer=custom_optimizer, loss=f1_loss, metrics=[f1_metric])


# Print model summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 densenet121 (Functional)    (None, 7, 7, 1024)        7037504   
                                                                 
 global_average_pooling2d (  (None, 1024)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 256)               262400    
                                                                 
 batch_normalization (Batch  (None, 256)               1024      
 Normalization)                                                  
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 2)                 5

In [ ]:
model.fit(train_generator, validation_data=val_generator, epochs=50, batch_size=batch_size, class_weight=class_weights_dict)

Epoch 1/50
38/38 [==============================] - 120s 3s/step - loss: -0.0219 - f1_metric: 0.5203 - val_loss: 0.0653 - val_f1_metric: 0.5939
Epoch 2/50
38/38 [==============================] - 98s 3s/step - loss: -0.0416 - f1_metric: 0.5377 - val_loss: 0.0575 - val_f1_metric: 0.6169
Epoch 3/50
38/38 [==============================] - 97s 3s/step - loss: -0.0449 - f1_metric: 0.5445 - val_loss: 0.0635 - val_f1_metric: 0.5316
Epoch 4/50
38/38 [==============================] - 94s 2s/step - loss: -0.0478 - f1_metric: 0.5466 - val_loss: 0.0662 - val_f1_metric: 0.5539
Epoch 5/50
38/38 [==============================] - 97s 3s/step - loss: -0.0495 - f1_metric: 0.5466 - val_loss: 0.0915 - val_f1_metric: 0.4780
Epoch 6/50
38/38 [==============================] - 97s 3s/step - loss: -0.0690 - f1_metric: 0.5778 - val_loss: 0.0752 - val_f1_metric: 0.5127
Epoch 7/50
38/38 [==============================] - 96s 3s/step - loss: -0.0602 - f1_metric: 0.5638 - val_loss: 0.1038 - val_f1_metric: 0.460

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import itertools

def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="black" if cm[i, j] > thresh else "red")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

def evaluate_binary_classification_model(model, test_generator):
    y_true = test_generator.classes
    y_pred = model.predict(test_generator, verbose=1)
    y_pred_labels = np.argmax(y_pred, axis=1)

    print("Classification Report:\n")
    print(classification_report(y_true, y_pred_labels, target_names=test_generator.class_indices.keys()))

    # Plot confusion matrix
    cnf_matrix = confusion_matrix(y_true, y_pred_labels)
    plot_confusion_matrix(cnf_matrix, classes=test_generator.class_indices.keys())

In [ ]:
evaluate_binary_classification_model(model, val_generator)